In [ ]:
from stable_baselines3.common.env_checker import check_env
from core.hydrodispatchenv import HydroDispatchEnv

env = HydroDispatchEnv(inflow_m3s=25)
# This checks:
# - observation_space and action_space are properly defined
# - reset() returns (obs, info) with obs in observation_space
# - step() returns (obs, reward, terminated, truncated, info)
# - observations stay within bounds
# - dtypes are correct (float32, not float64! or not array(float))
print("Running Gymnasium API check...")
check_env(env, warn=True)
print("✅ Environment passed API validation!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

env = HydroDispatchEnv(inflow_m3s=10000.0)
obs, info = env.reset(seed=42)

# Track everything
data = {
    "volumes": [], "rewards": [], "powers": [],
    "discharges": [], "spills": [], "tariffs": [],
    "obs_min": [], "obs_max": [],
}

for step in range(env.HOURS_PER_EPISODE):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)

    data["volumes"].append(info["volume_m3"])
    data["rewards"].append(reward)
    data["powers"].append(info["power_mw"])
    data["discharges"].append(info["actual_discharge_m3s"])
    data["spills"].append(info["spill_m3s"])
    data["obs_min"].append(obs.min())
    data["obs_max"].append(obs.max())

    if terminated or truncated:
        break

# === SANITY CHECKS ===
print("\n=== ENVIRONMENT SANITY CHECKS ===")

# Check 1: Observations stay in bounds
obs_violations = sum(1 for mn, mx in zip(data["obs_min"], data["obs_max"])
                     if mn < -0.01 or mx > 5.01) # Updated bound for dynamic tariff
print(f"Observation bound violations: {obs_violations} "
      f"({'✅ PASS' if obs_violations == 0 else '❌ FAIL'})")

# Check 2: Volume never exceeds physical bounds
vol_violations = sum(1 for v in data["volumes"]
                     if v < env.RESERVOIR_MIN_M3 - 1 or v > env.RESERVOIR_MAX_M3 + 1)
print(f"Volume bound violations:      {vol_violations} "
      f"({'✅ PASS' if vol_violations == 0 else '❌ FAIL'})")

# Check 3: Power is never negative
neg_power = sum(1 for p in data["powers"] if p < -0.01)
print(f"Negative power events:        {neg_power} "
      f"({'✅ PASS' if neg_power == 0 else '❌ FAIL'})")

# Check 4: Discharge never exceeds Q_MAX
over_discharge = sum(1 for d in data["discharges"]
                     if d > env.TURBINE_Q_MAX + 0.01)
print(f"Over-discharge events:        {over_discharge} "
      f"({'✅ PASS' if over_discharge == 0 else '❌ FAIL'})")

print(f"\nTotal reward (random agent):  ${sum(data['rewards']):,.0f}")
print(f"Mean reward per step:         ${np.mean(data['rewards']):,.1f}")

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 12))

hours = range(len(data["volumes"]))

# Reservoir level
axes[0,0].plot(hours, [v/1e6 for v in data["volumes"]], 'b-')
axes[0,0].axhline(y=env.RESERVOIR_MAX_M3/1e6, color='r', ls='--', alpha=0.5)
axes[0,0].axhline(y=env.RESERVOIR_MIN_M3/1e6, color='orange', ls='--', alpha=0.5)
axes[0,0].set_title("Reservoir Volume (M m³)")

# Discharge
axes[0,1].plot(hours, data["discharges"], 'g-', alpha=0.7)
axes[0,1].axhline(y=25.0, color='blue', ls=':', label='Inflow')
axes[0,1].set_title("Actual Discharge (m³/s)")
axes[0,1].legend()

# Power
axes[1,0].plot(hours, data["powers"], 'purple', alpha=0.7)
axes[1,0].set_title("Power Generated (MW)")

# Reward
axes[1,1].plot(hours, data["rewards"], 'red', alpha=0.7)
axes[1,1].set_title(f"Reward per Step (Total: ${sum(data['rewards']):,.0f})")

# Cumulative reward
axes[2,0].plot(hours, np.cumsum(data["rewards"]), 'darkred', linewidth=2)
axes[2,0].set_title("Cumulative Reward ($)")
axes[2,0].set_xlabel("Hour")

# Spill events
axes[2,1].bar(hours, data["spills"], color='orange', alpha=0.7)
axes[2,1].set_title("Spill Events (m³/s)")
axes[2,1].set_xlabel("Hour")

plt.suptitle("HydroDispatchEnv: Random Agent Diagnostic Dashboard", fontsize=14)
plt.tight_layout()
plt.show()